# AML Signal Escalation — Submission Pipeline

Track A (feature engineering) va Track B (modeling) qismlarini boshidan oxirigacha birlashtiruvchi to'liq qayta ishga tushiriluvchi (reproducible) notebook.
Jamoa ID: **C6FD20A0** | Baholash metrikasi: **ROC-AUC**

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

TEAM_ID = "C6FD20A0"

from src import config
from src.data_loading import load_signals, load_transactions
from src.features import build
from src.model import train, cross_validate
from src.predict import predict, validate_submission, write

## 1. Load raw data

In [2]:
train_signals = load_signals(config.TRAIN_SIGNALS_PATH)
train_transactions = load_transactions(config.TRAIN_TRANSACTIONS_PATH)
test_signals = load_signals(config.TEST_SIGNALS_PATH)
test_transactions = load_transactions(config.TEST_TRANSACTIONS_PATH)
train_signals.shape, train_transactions.shape, test_signals.shape, test_transactions.shape

((14000, 3), (6987663, 5), (6000, 2), (3027575, 5))

## 2. Build features (Track A: 25 domain features, 0% lookahead leakage)

In [3]:
train_features = build(train_signals, train_transactions)
test_features = build(test_signals, test_transactions)
train_features.head(3)

,signal_id,n_txn,amt_mean,amt_std,amt_max,amt_sum,frac_kirim,frac_karta,frac_bank_otkazmasi,frac_naqd,frac_xalqaro,frac_night,frac_weekend,frac_extreme,n_txn_1d,n_txn_7d,n_txn_30d,span_days,velocity,hour_entropy,hour_maxshare,dow_entropy,dow_maxshare,frac_kirim_1d,ratio_n_1d_to_7d,amt_mean_1d,eskalatsiya
0,SG_000002,203,0.437551,1.283484,3.816146,88.822846,0.665025,0.379310,0.546798,0.073892,0.00000,0.216749,0.369458,0.137931,25.0,33.0,76,116.957928,1.720953,3.041451,0.128079,1.901236,0.246305,0.720000,5.192878,0.595016,0
1,SG_000003,271,0.327211,0.714104,3.806999,88.674149,0.675277,0.254613,0.642066,0.095941,0.00738,0.210332,0.357934,0.014760,18.0,23.0,39,178.943530,1.506028,3.102211,0.103321,1.916735,0.225092,0.722222,5.316456,0.440592,0
2,SG_000004,760,0.025983,0.997670,4.498756,19.747271,0.828947,0.622368,0.357895,0.019737,0.00000,0.225000,0.243421,0.048684,77.0,86.0,150,179.814097,4.203212,3.097765,0.134211,1.914601,0.226316,0.883117,6.216840,-0.112124,0


## 3. Cross-validate and train model (Track B: Calibrated Logistic Regression)

In [4]:
X = train_features[config.FEATURE_COLUMNS]
y = train_features[config.TARGET_COL]
auc = cross_validate(X, y)
print(f"CV ROC-AUC: {auc:.4f}")
model = train(X, y)

CV ROC-AUC: 0.5666


## 4. Predict + write submission (outputs/team_C6FD20A0.csv)

In [5]:
predictions = predict(model, test_features)
validate_submission(predictions, test_signals[config.ID_COL])
out_path = str(config.OUTPUTS_DIR / f"team_{TEAM_ID}.csv")
write(predictions, out_path)
print(f"wrote {out_path}")
predictions.head()

wrote C:\Users\Windows_11\Desktop\fintech_track_data\outputs\team_C6FD20A0.csv


,signal_id,ehtimollik
0,SG_000001,0.169658
1,SG_000007,0.192778
2,SG_000009,0.141858
3,SG_000010,0.126994
4,SG_000011,0.193400
